In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df=pd.read_csv("Telco-Customer-Churn.csv")
print("Shape:",df.shape)
print(df.columns.tolist())
df.info()
df.head(5)
#df.describe()

In [ ]:
# Check how many bad values exist
problem = df[df['TotalCharges'] == ' ']
print("Rows with space in TotalCharges:", len(problem))
print(problem[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

df['TotalCharges']=df['TotalCharges'].replace(' ',np.nan)
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')

print(df['TotalCharges'].dtype)
print(df['TotalCharges'].isnull().sum())

In [ ]:
missing=df[df['TotalCharges'].isnull()]
print("missing values",len(missing))
print(missing[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])


In [ ]:
print(df['Churn'].value_counts())
print(df['Churn'].dtype)
df['Churn']=df['Churn'].map({'Yes':1,'No':0})
print(df['Churn'].value_counts())
print(df['Churn'].dtype)

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nChurn distribution:\n", df['Churn'].value_counts())
print("\nChurn %:\n", df['Churn'].value_counts(normalize=True).round(3) * 100)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))

axes[0].hist(df['tenure'],bins=20,edgecolor='black',color='lavender')
axes[0].set_title('Tenure Distribution')
axes[0].set_xlabel('Months')
axes[0].set_ylabel('Count')

axes[1].hist(df['MonthlyCharges'],bins=20,edgecolor='black',color='steelblue')
axes[1].set_title('Monthly Charges Distribution')
axes[1].set_xlabel('Amount ($)')

axes[2].hist(df['TotalCharges'],bins=20,edgecolor='black',color='green')
axes[2].set_title('Total Charges Distribution')
axes[2].set_xlabel('Amount ($)')

plt.suptitle('Numeric Columns Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['gender', 'SeniorCitizen', 'Partner',
            'Dependents', 'PhoneService', 'Contract',
            'PaymentMethod', 'InternetService']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i,col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar',ax=axes[i],edgecolor='black',color='yellow')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)#for big text to read clean
plt.suptitle('Categorical Columns Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# tenure vs churn
axes[0].hist(df[df['Churn']==0]['tenure'],
             bins=30, alpha=0.7,
             color='steelblue', label='Stayed')
axes[0].hist(df[df['Churn']==1]['tenure'],
             bins=30, alpha=0.7,
             color='tomato', label='Churned')
axes[0].set_title('Tenure vs Churn')
axes[0].set_xlabel('Months')
axes[0].set_ylabel('Count')
axes[0].legend()

# MonthlyCharges vs churn
axes[1].hist(df[df['Churn']==0]['MonthlyCharges'],
             bins=30, alpha=0.7,
             color='steelblue', label='Stayed')
axes[1].hist(df[df['Churn']==1]['MonthlyCharges'],
             bins=30, alpha=0.7,
             color='tomato', label='Churned')
axes[1].set_title('Monthly Charges vs Churn')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

# TotalCharges vs churn
axes[2].hist(df[df['Churn']==0]['TotalCharges'],
             bins=30, alpha=0.7,
             color='steelblue', label='Stayed')
axes[2].hist(df[df['Churn']==1]['TotalCharges'],
             bins=30, alpha=0.7,
             color='tomato', label='Churned')
axes[2].set_title('Total Charges vs Churn')
axes[2].set_xlabel('Amount ($)')
axes[2].legend()

plt.suptitle('Numeric Columns vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols2 = ['gender', 'SeniorCitizen', 'Partner',
             'Dependents', 'Contract',
             'PaymentMethod', 'InternetService', 'TechSupport']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols2):
    churn_rate = df.groupby(col)['Churn'].mean() * 100

    bars = axes[i].bar(churn_rate.index,
                       churn_rate.values,
                       color='tomato',
                       edgecolor='black')

    for bar in bars:
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     height + 0.5,
                     f'{height:.1f}%',
                     ha='center', fontsize=9)

    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Churn %')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Churn Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].corr()
print(corr.round(2))

In [ ]:
print("""
==============================================
   TELCO CUSTOMER CHURN — EDA CONCLUSIONS
==============================================

DATASET:
→ 7043 customers, 21 columns
→ 26.5% customers churned (1869 out of 7043)

TOP CHURN DRIVERS FOUND:
─────────────────────────────────────────
1. CONTRACT TYPE
   Month-to-month → 42.7% churn
   Two year       →  2.8% churn
   No commitment = easy to leave

2. PAYMENT METHOD
   Electronic check → 45.3% churn
   Auto payment     → 15-16% churn
   Manual payment = customer thinks about cost monthly

3. INTERNET SERVICE
   Fiber optic → 41.9% churn
   DSL         → 19.0% churn
   Premium price not matching expected quality

4. SENIOR CITIZENS
   Seniors     → 41.7% churn
   Non-seniors → 23.6% churn
   Technology complexity + different options available

5. TECH SUPPORT
   No support  → 41.6% churn
   Has support → 15.2% churn
   Problems without help = customer leaves

6. TENURE
   Correlation → -0.35
   New customers churn most
   After 2 years customers become very loyal

7. MONTHLY CHARGES
   Correlation → +0.19
   Higher paying customers churn more
   Value for money is questioned

WHAT DOES NOT AFFECT CHURN:
─────────────────────────────────────────
→ Gender (26.9% vs 26.2% — almost equal)

BUSINESS RECOMMENDATIONS:
─────────────────────────────────────────
1. Push customers toward annual/two year contracts
   even with small discounts — saves more in long run

2. Encourage auto payment setup
   reduces manual billing friction

3. Investigate fiber optic quality issues
   or reprice fiber optic plans

4. Create special simplified plans for seniors

5. Make tech support available to all customers
   especially in first 12 months

6. Focus retention efforts on NEW customers
   first 1-2 years are the most critical period
==============================================
""")